# Blur Core

Blur Core is the engine under every Blur media engine. It takes any grid of non-negative numbers (pixels, notes, samples, a volume), loads it into one quantum state, rotates each qubit once and reads the grid back. Values spread through interference rather than averaging, so weight jumps to places a convolution kernel would never reach.

Blur Image, Teleblur and Blur MIDI wrap this engine for one kind of file. Here you work with the numbers directly.

## Setup

In [ ]:
import getpass
import os

import matplotlib.pyplot as plt
import numpy as np

from moth_atlas import Atlas

ENGINE = "blur-core-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "blur-core-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
def grid(job):
    """The blurred grid a job returned, as a numpy array the shape of `values`."""
    result = job.result
    return np.asarray(result["values"] if isinstance(result, dict) else result, dtype=float)


def bars(rows):
    """Plot {label: 1-D values} as bar charts, one row each."""
    fig, axes = plt.subplots(len(rows), 1, figsize=(8, 1.3 * len(rows)), sharex=True, squeeze=False)
    for ax, (label, values) in zip(axes[:, 0], rows.items()):
        ax.bar(range(len(values)), values, color="C0")
        ax.set_ylabel(label, rotation=0, ha="right", va="center")
        ax.set_yticks([])
    axes[-1, 0].set_xlabel("index")
    plt.show()


def pictures(grids, size=2.2):
    """Show {label: 2-D grid} side by side."""
    fig, axes = plt.subplots(1, len(grids), figsize=(size * len(grids), size + 0.4), squeeze=False)
    for ax, (label, g) in zip(axes[0], grids.items()):
        ax.imshow(g, cmap="magma", vmin=0)
        ax.set_title(label, fontsize=9)
        ax.set_axis_off()
    plt.show()

## Try it

A single spike in a row of 16 zeros: 16 values fit in 4 qubits. `run` submits the job and waits; `job.result` holds the blurred grid.

In [ ]:
pulse = [0.0] * 16
pulse[5] = 1.0

job = client.jobs.run(
    engine=ENGINE,
    params={
        "values": pulse,
        "strength": 0.5,
        "reach": 0,
        "style": "x",
    },
)
bars({"input": pulse, "blurred": grid(job)})

## Parameters

What to expect from each:

- `values`: a list, or nested lists, of non-negative numbers. The nesting depth is the grid's dimensionality; it must be rectangular.
- `strength` (0 to 1): how far each value turns. 0 returns the input unchanged, 1 is the strongest blur. A list sets it per axis, so one axis can stay sharp while another blurs.
- `reach` (0 to 1): 0 lets only neighbouring grid points mix; 1 lets any point affect any other equally.
- `style`: which rotations each qubit gets, in order: `x`, `y` or a combination such as `xy`. `x` spreads weight evenly; `y` holds it on edges.
- `axes`: which dimensions to blur, e.g. `[1]` for only the columns of a 2-D grid. Leave it out to blur all of them.
- `shots`: leave it out for the exact result; a number recovers the grid from that many simulated measurements, with shot noise.
- `max_qubits` (up to 24): a safety cap on the grid size. A 1024 × 1024 grid needs 20 qubits.

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Encode.** Every grid point gets a basis state, and its value sets how much weight that state carries: the amplitude is $\sqrt{v}$, normalised over the grid. Indices are Gray-coded per axis, so neighbouring points differ in exactly one bit. A grid of shape $(N_1, N_2, \dots)$ needs

$$n = \sum_k \lceil \log_2 N_k \rceil \text{ qubits.}$$

**Rotate.** Each qubit of a blurred axis gets one `rx` and/or `ry`, at an angle set by `strength`. Because neighbours differ in one bit, a rotation on that bit moves weight between them. With `reach` near 0 the angles favour local moves; at 1 every qubit turns equally and weight can land anywhere.

**Read back.** The probabilities $|\text{amp}|^2$ are the new grid, rescaled to the input's maximum. At `strength` 0 the round trip is exact. Totals aren't preserved exactly.

The interference is the point: several rotations reach the same grid point by different paths, and those paths add or cancel. That's where Blur's echoes come from.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Strength

At 0 the spike comes back as it went in. As strength grows, weight leaves the spike and lands on the points one Gray-code step away, not just its immediate left and right.

In [ ]:
rows = {}
for strength in (0.0, 0.25, 0.5, 0.75, 1.0):
    print(f"strength={strength}")
    job = client.jobs.run(engine=ENGINE, params={"values": pulse, "strength": strength}, quiet=True)
    rows[f"strength {strength}"] = grid(job)
bars(rows)

### Reach

At the same strength, `reach` decides where the weight goes: near neighbours at 0, anywhere at 1.

In [ ]:
rows = {}
for reach in (0.0, 0.5, 1.0):
    print(f"reach={reach}")
    job = client.jobs.run(engine=ENGINE, params={"values": pulse, "strength": 0.5, "reach": reach}, quiet=True)
    rows[f"reach {reach}"] = grid(job)
bars(rows)

### Axes and per-axis strength

A 2-D grid: a ring on a 32 × 32 canvas (10 qubits). Blurring only axis 0 smears it vertically, only axis 1 horizontally. A strength list sets each axis on its own.

In [ ]:
yy, xx = np.mgrid[:32, :32]
ring = (np.abs(np.hypot(xx - 15.5, yy - 15.5) - 9) < 2).astype(float)

grids = {"input": ring}
for label, extra in [("both axes", {}), ("axis 0 only", {"axes": [0]}), ("axis 1 only", {"axes": [1]}),
                     ("strength [0.8, 0.1]", {"strength": [0.8, 0.1]})]:
    params = {"values": ring.tolist(), "strength": 0.6, **extra}
    print({k: v for k, v in params.items() if k != "values"})
    grids[label] = grid(client.jobs.run(engine=ENGINE, params=params, quiet=True))
pictures(grids)

### Style

`x` spreads the ring's weight across the canvas; `y` keeps it on the ring's edges; `xy` applies both, one after the other.

In [ ]:
grids = {"input": ring}
for style in ("x", "y", "xy"):
    print(f"style={style}")
    job = client.jobs.run(engine=ENGINE, params={"values": ring.tolist(), "strength": 0.6, "style": style}, quiet=True)
    grids[style] = grid(job)
pictures(grids)

### Shots

Without `shots` the result is exact. With them, the grid is rebuilt from that many measurements, as it would be on hardware, and grain appears where weight is thin.

In [ ]:
grids = {}
for shots in (None, 4096, 512):
    print(f"shots={shots}")
    params = {"values": ring.tolist(), "strength": 0.6, **({"shots": shots} if shots else {})}
    grids[f"shots {shots or 'exact'}"] = grid(client.jobs.run(engine=ENGINE, params=params, quiet=True))
pictures(grids)

## Questions

**Which Blur engine should I use?**

Blur Core for anything you can turn into numbers; Blur Image for an image file, Teleblur to morph one image into another, Blur MIDI for a MIDI file. The media engines handle decoding, masks and tiling for you.

**Why must values be non-negative?**

Each value becomes a probability weight. To blur signed data such as audio samples, shift it into [0, 1], blur, rescale to the input's total and shift back.

**How big can a grid be?**

Qubits grow with the log of each axis: 1024 × 1024 is 20 qubits, and `max_qubits` allows up to 24. Simulation time grows exponentially with qubits, so split larger data into tiles and blur them one run each.

**What about colour?**

Blur each channel as its own grid and recombine them; that's what Blur Image does.

**Is this running on quantum hardware?**

No: on a statevector simulator, so results are exact and repeatable. `shots` shows what sampling on hardware would add.